In [18]:
# APPENDIX 1
from pathlib import Path
import numpy as np
import pandas as pd

# Set student-specific parameters
LAST_NAME = "Bacor"
STUDENT_ID = "TUPM-26-1879"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, "Student ID must contain digits"

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)
verification_value = ID_SUM * NAME_LENGTH + SEED_DIGIT

# Setup the directory paths.
WORK_DIR = Path.cwd()
DATA_DIR = Path.cwd().parent / "data"
OUTPUT_DIR = WORK_DIR / "output"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"
description_path = DATA_DIR / "secom.names"

# Verify if files are missing
for path in [measurement_path, label_path, description_path]:
    assert path.is_file(), f"Missing file: {path}"

print("Workspace:", WORK_DIR.resolve())
print("All dataset files verified")

raw_measurements = np.genfromtxt(measurement_path, dtype=float)

raw_labels = np.genfromtxt(label_path, dtype=float)
preserved_verify = raw_labels.shape[0] == raw_measurements.shape[0]

print("Measurements and label correspondence are preserved?", preserved_verify)

# Set the random seed for reproducible result.
rng = np.random.default_rng(RANDOM_SEED)

# Identify missing and constant values 
missing = np.isnan(raw_measurements)
observed = (~missing).sum(axis=0)

all_missing = observed == 0

observed_min = np.where(
    missing, np.inf, raw_measurements
).min(axis=0)

observed_max = np.where(
    missing, -np.inf, raw_measurements
).max(axis=0)

constant = (observed > 0) & (observed_min == observed_max)

# Find out the raw usable records and figure out the needed parameters.
usable_indices = np.flatnonzero(~(all_missing | constant))

raw_usable = raw_measurements[:, usable_indices].copy()

feature_count = (5 + (SEED_DIGIT % 3))

feature_positions = rng.choice(
    raw_usable.shape[1], size=feature_count, replace=False
)

assigned_features = raw_usable[:, feature_positions]

feature_means = np.nanmean(assigned_features, axis=0)
feature_sds = np.nanstd(assigned_features, axis=0, ddof=1)
feature_medians = np.nanmedian(assigned_features, axis=0)
selected_original_indices = usable_indices[feature_positions]
assigned_feature_names = [f"Feature_{i}" for i in selected_original_indices]
# generated by ChatGPT, prompt: "is my method of finding the highest missing percentage correct? the data is 2D"
missing_percentage = (missing.sum(axis=0) / raw_measurements.shape[0]) * 100
highest_missing_prcntg = missing_percentage.max()


# Standardization of values using Median and MAD
prepared_np = np.where(
    np.isnan(assigned_features), feature_medians, assigned_features
)

# Calculate the mean and median, generated by: Gemini, prompt: "how can i rewrite it into MAD?"
median = np.median(prepared_np, axis=0)
abs_dev = (prepared_np - median)
mad = np.median(np.abs(abs_dev), axis=0)

# Scale MAD by 1.4826 if MAD is above 0, otherwise scale by 1.2533 if MAD is 0
scale = np.where(mad > 0, 1.4826 * mad, feature_sds)
safe_scale = np.where(scale > 0, scale, 1.0)

# Set the modified z-score
standardized_np = (prepared_np - median) / safe_scale
flags = np.abs(standardized_np) > 3.5

flagged_percentage = (flags.sum(axis=0) / flags.shape[0]) * 100
sorted_flagged = np.sort(flagged_percentage)

# Ranking and exporting generated by ChatGPT, prompt: "how can i rank the features by their percentage of flagged observations"
# Ranking features based on their percentage of flagged observations.
ranked_indices = np.argsort(flagged_percentage)[::-1]

# Arrange the percentages according to their ranking
ranked_percentages = flagged_percentage[ranked_indices]

ranking_array = np.column_stack((
    np.arange(1, len(ranked_indices) + 1),
    selected_original_indices[ranked_indices],
    ranked_percentages,
    feature_means[ranked_indices],
    feature_medians[ranked_indices],
    feature_sds[ranked_indices],
    missing_percentage[selected_original_indices[ranked_indices]],
    mad[ranked_indices]
    
))

# Save to CSV
np.savetxt(
    OUTPUT_DIR / f"measurement_quality.csv",
    ranking_array,
    delimiter=",",
    header="Rank,Feature_Index,Flagged_Percentage,Feature-wise Mean,Feature-wise Median,Feature-wise Sample SD,Missing Percentage,MAD",
    comments="",
    fmt=["%d", "%d", "%.4f", "%.4f", "%.4f", "%.4f", "%.4f", "%.4f"]
)

# Verification: 
# Verify rankings
ranking_verified = np.all(flagged_percentage[ranked_indices][:-1] >= flagged_percentage[ranked_indices][1:])


manual_flagged_percentage = []
# Verify Flagged Percentage
for col in range(flags.shape[1]):
    flagged_count = 0 
    for row in range(flags.shape[0]):
        if flags[row, col]:
            flagged_count += 1
    percentage = (flagged_count / flags.shape[0]) * 100
    manual_flagged_percentage.append(percentage)

flagged_percentage_verify = np.allclose(flagged_percentage, manual_flagged_percentage)

highest_flagged_verify = ranked_indices[0]

manual_highest_index = 0
for i in range(1, len(manual_flagged_percentage)):
    if manual_flagged_percentage[i] > manual_flagged_percentage[manual_highest_index]:
        manual_highest_index = i

highest_flagged_verify = (ranked_indices[0] == manual_highest_index)

# Verify Flagged Total
manual_total_flagged = 0
for col in range(flags.shape[1]):
    for row in range(flags.shape[0]):
        if flags[row, col]:
            manual_total_flagged += 1

total_flagged_verify = (manual_total_flagged == np.count_nonzero(flags))

# Verify Missing Percentage
manual_missing_percentage = []
for col in range(missing.shape[1]):
    missing_count = 0 
    for row in range(missing.shape[0]):
        if missing[row, col]:
            missing_count += 1
    percentage = (missing_count / raw_measurements.shape[0]) * 100
    manual_missing_percentage.append(percentage)

manual_missing_percentage = np.array(manual_missing_percentage)

manual_highest_missing = 0
for i in range(1, len(manual_missing_percentage)):
    if manual_missing_percentage[i] > manual_missing_percentage[manual_highest_missing]:
        manual_highest_missing = i
manual_highest_missing = max(manual_missing_percentage)
total_missing_verify = np.isclose(
    manual_highest_missing, 
    highest_missing_prcntg
)

# Verification of Feature-wise means, generated by ChatGPT
manual_means = []
manual_med = []
manual_sds = []
for col in range(assigned_features.shape[1]):
    values = assigned_features[:, col]
    values = values[~np.isnan(values)]

    values.sort()
    n = len(values)

    if n % 2 == 0:
        med = (values[n//2 - 1] + values[n//2]) / 2
    else:
        med = values[n//2]

    mean = np.sum(values) / len(values)

    variance = np.sum((values - mean) ** 2) / (len(values) - 1)
    manual_sds.append(np.sqrt(variance))
    manual_med.append(med)
    manual_means.append(np.sum(values) / n)
manual_medians = np.array(manual_med)
manual_sds = np.array(manual_sds)
manual_means = np.array(manual_means)
mean_verify = np.allclose(manual_means, feature_means)
median_verify = np.allclose(manual_medians, feature_medians)
sds_verify = np.allclose(manual_sds, feature_sds)
missing_percentage_verify = np.allclose(missing_percentage, manual_missing_percentage)
vectorized_results = all(
    [mean_verify, median_verify, sds_verify, flagged_percentage_verify, highest_flagged_verify, total_flagged_verify, total_missing_verify, missing_percentage_verify, ranking_verified]
)



# Printing
print("Selected Feature Positions:", assigned_feature_names)
print("First Feature Mean:", f"{feature_means[0]:.4f}")
print("First Feature Median:",  f"{feature_medians[0]:.4f}")
print("First Feature Sample SD:",  f"{feature_sds[0]:.4f}")
print("Highest Missing Percentage:", f"{round(highest_missing_prcntg, 4)}%")
print("Total Flagged Observations:", np.count_nonzero(flags))
print("Feature with Highest Flagged Percentage:", f"Feature_{selected_original_indices[ranked_indices][0]}")
print("Measurements with Absolute Robust Standardized Values ±3.5:", np.count_nonzero(flags))
print()
print("=" * 50)
print("Vectorized Results Verification:", vectorized_results)
print("Feature-wise Mean Verified?", mean_verify)
print("Feature-wise Median Verified?", median_verify)
print("Feature-wise Standard Deviation Verified?", sds_verify)
print("Flagged Percentage Verified?", flagged_percentage_verify)
print("Highest Flagged Percentage Verified?", highest_flagged_verify)
print("Total Flagged Observations Verified?", total_flagged_verify)
print("Missing Percentage Verified?", missing_percentage_verify)
print("Highest Missing Percentage Verified?", total_missing_verify)










Workspace: C:\Users\hp\Lab7_NumPy_Pandas\Lab7_NumPy_Pandas_Additional_Exercises
All dataset files verified
Measurements and label correspondence are preserved? True
Selected Feature Positions: ['Feature_160', 'Feature_77', 'Feature_51', 'Feature_473', 'Feature_453']
First Feature Mean: 555.3463
First Feature Median: 438.0000
First Feature Sample SD: 574.8086
Highest Missing Percentage: 91.1934%
Total Flagged Observations: 122
Feature with Highest Flagged Percentage: Feature_160
Measurements with Absolute Robust Standardized Values ±3.5: 122

Vectorized Results Verification: True
Feature-wise Mean Verified? True
Feature-wise Median Verified? True
Feature-wise Standard Deviation Verified? True
Flagged Percentage Verified? True
Highest Flagged Percentage Verified? True
Total Flagged Observations Verified? True
Missing Percentage Verified? True
Highest Missing Percentage Verified? True


In [19]:
# APPENDIX 2
rng = np.random.default_rng(RANDOM_SEED)
n_features = raw_measurements.shape[1]
feature_names = [f"Feature_{i:03d}" for i in range(n_features)]

sample_path = OUTPUT_DIR / "balanced_sample.csv"
feature_path = OUTPUT_DIR / "feature_comparison.csv"

df = pd.DataFrame(
    raw_measurements.copy(), columns=feature_names
)

labels = np.loadtxt(label_path, usecols=0, dtype=int)

assert len(labels) == len(df)
assert np.isin(labels, [-1, 1]).all()

df["Result"] = labels
df["Outcome"] = np.where(
    labels == -1, "Pass", "Fail"
)

pass_count = (labels == -1).sum()
fail_count = (labels == 1).sum()

# Generated by ChatGPT, prompt: "on this code snippet, how can i produce a reproducible balanced sample containing min(20 + ID_SUM, Pass count, Fail count) records per outcome"
# Determine the number of records to sample per outcome
sample_size = min(20 + ID_SUM, pass_count, fail_count)

df["_Original_Index"] = np.arange(len(df))
# Create a reproducible balanced sample
balanced_df = (
    df.groupby("Outcome", group_keys=False)
      .sample(n=sample_size, random_state=RANDOM_SEED)
      .reset_index(drop=True)
)

balanced_indices = balanced_df["_Original_Index"].to_numpy()

balanced_df.to_csv(sample_path, index=False)
print("Balanced File Successfully Created:", sample_path.exists())

without_all_missing = balanced_df.dropna(axis=1, how="all")
retained_features = [
    feature for feature in feature_names
    if feature in without_all_missing.columns
]
feature_positions = rng.choice(
    len(retained_features), size=5, replace=False
)
full_median = df[retained_features].median()
clean = without_all_missing[retained_features].copy()
clean = clean.fillna(full_median)

clean["Result"] = balanced_df["Result"].to_numpy()
clean["Outcome"] = balanced_df["Outcome"].to_numpy()
clean["Is_Fail"] = balanced_df["Result"] == 1

selected_features = [
    retained_features[i] for i in feature_positions
]

row_positions = rng.choice(
    len(clean), size=5, replace=False
)
sample = clean.iloc[row_positions][
    selected_features + ["Result", "Outcome", "Is_Fail"]
].copy()


amd_smd_results = []
for feature in selected_features:

    # Separate Pass and Fail measurements
    pass_values = clean.loc[
        clean["Outcome"] == "Pass", feature
    ]

    fail_values = clean.loc[
        clean["Outcome"] == "Fail", feature
    ]

    # Group counts
    pass_count_feature = pass_values.count()
    fail_count_feature = fail_values.count()

    # Group means and medians
    pass_mean = pass_values.mean()
    fail_mean = fail_values.mean()

    pass_median = pass_values.median()
    fail_median = fail_values.median()

    # Sample standard deviations
    pass_std = pass_values.std(ddof=1)
    fail_std = fail_values.std(ddof=1)

    # Absolute Mean Difference
    mean_diff = pass_mean - fail_mean
    amd = abs(mean_diff)

    # Pooled within-group sample standard deviation
    pooled_std = np.sqrt(
        (
            (pass_count_feature - 1) * pass_std**2
            + (fail_count_feature - 1) * fail_std**2
        ) /
        (
            pass_count_feature + fail_count_feature - 2
        )
    ) if pass_count_feature + fail_count_feature > 2 else np.nan

    # Standardized Mean Difference
    smd = (
        mean_diff / pooled_std
        if pooled_std != 0 and np.isfinite(pooled_std)
        else np.nan
    )

    abs_smd = abs(smd)

    amd_smd_results.append({
        "Feature": feature,

        "Pass Count": pass_count_feature,
        "Pass Mean": pass_mean,
        "Pass Median": pass_median,
        "Pass SD": pass_std,

        "Fail Count": fail_count_feature,
        "Fail Mean": fail_mean,
        "Fail Median": fail_median,
        "Fail SD": fail_std,

        "AMD": amd,
        "SMD": smd,
        "ABS_SMD": abs_smd
    })

amd_smd_summary = pd.DataFrame(amd_smd_results)

profiling_summary = (
    sample[selected_features]
    .agg(["count", "mean", "median", "std"])
    .T.rename_axis("Feature")
    .reset_index()
    .sort_values(by="std", ascending=False)
)

combined_summary = pd.merge(
    profiling_summary,
    amd_smd_summary,
    on="Feature",
    how="left"
)

combined_summary = combined_summary.sort_values(
    by="ABS_SMD",
    ascending=False
).reset_index(drop=True)

# Exporting CSV
combined_summary.to_csv(feature_path, index=False)
print("Feature Comparison Successfully Exported:", feature_path.exists())

# NUMPY VERIFICATION, generated by: ChatGPT

balanced_raw = raw_measurements[balanced_indices]

# Select the same five features
selected_positions = [
    feature_names.index(feature)
    for feature in selected_features
]

selected_np = balanced_raw[:, selected_positions]

# Get the same features from the full dataset
full_selected_np = raw_measurements[:, selected_positions]

# Calculate medians using the full dataset
feature_medians = np.nanmedian(full_selected_np, axis=0)

# Apply median imputation
prepared_np = np.where(
    np.isnan(selected_np),
    feature_medians,
    selected_np
)

# Compare Pandas and NumPy results
pandas_sd = clean[selected_features[0]].std(ddof=1)
numpy_sd = prepared_np[:, 0].std(ddof=1)

clean_match = np.allclose(
    clean[selected_features].to_numpy(),
    prepared_np
)

sd_match = np.isclose(pandas_sd, numpy_sd)
match = all([clean_match, sd_match])

print()
print("--- SAMPLE STATISTICS ---")
print(sample.head())
print()
print("=" * 50)
print()
print("--- GROUPED STATISTICS ---")
print(combined_summary.round(4))
print("\nBalanced Sample Size:", len(balanced_df))
print("Selected Features:", selected_features)
print("Highest-Ranked Feature:", combined_summary.iloc[0]["Feature"])
print("Absolute Standardized Mean Difference", [float(item["ABS_SMD"]) for item in amd_smd_results])
print("NumPy-Pandas Statistics Match:", match)



Balanced File Successfully Created: True
Feature Comparison Successfully Exported: True

--- SAMPLE STATISTICS ---
    Feature_189  Feature_089  Feature_059  Feature_477  Feature_462  Result  \
52          0.0       0.1691       1.1655       3.6022          0.0       1   
32          0.0       0.1674      -2.5418       4.6392          0.0       1   
84          0.0       0.2035       8.1418       5.2820          0.0      -1   
92          0.0       0.2179       3.3691       5.0427          0.0      -1   
46          0.0       0.2094      16.6491      12.4472          0.0       1   

   Outcome  Is_Fail  
52    Fail     True  
32    Fail     True  
84    Pass    False  
92    Pass    False  
46    Fail     True  


--- GROUPED STATISTICS ---
       Feature  count    mean  median     std  Pass Count  Pass Mean  \
0  Feature_059    5.0  5.3567  3.3691  7.4023          53     2.8462   
1  Feature_477    5.0  6.2027  5.0427  3.5494          53     6.9565   
2  Feature_089    5.0  0.1935  0.

C:\Users\hp\AppData\Local\Temp\ipykernel_14880\1737412112.py:55: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  clean["Result"] = balanced_df["Result"].to_numpy()
C:\Users\hp\AppData\Local\Temp\ipykernel_14880\1737412112.py:56: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  clean["Outcome"] = balanced_df["Outcome"].to_numpy()
C:\Users\hp\AppData\Local\Temp\ipykernel_14880\1737412112.py:57: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performanc

In [20]:
# APPENDIX 3
rng = np.random.default_rng(RANDOM_SEED)
csv_path = OUTPUT_DIR / "robustness_comparison.csv"
report_path = OUTPUT_DIR / "sensitivity_report.txt"

n_features = raw_measurements.shape[1]
feature_names = [f"Feature_{i:03d}" for i in range(n_features)]
df = pd.DataFrame(
    raw_measurements.copy(), columns=feature_names
)

labels = np.loadtxt(label_path, usecols=0, dtype=int)

assert len(labels) == len(df)
assert np.isin(labels, [-1, 1]).all()

df["Result"] = labels
df["Outcome"] = np.where(
    labels == -1, "Pass", "Fail"
)

# Determining of observed measurements, and usable features generated by ChatGPT
pass_observed = (
    df.loc[df["Outcome"] == "Pass", feature_names]
    .count()
)

fail_observed = (
    df.loc[df["Outcome"] == "Fail", feature_names]
    .count()
)
# Find out the raw usable records and figure out the needed parameters.
retained_features = [
    feature for feature in feature_names
    if pass_observed[feature] >= 10
    and fail_observed[feature] >= 10
]

retained_features = [
    feature for feature in retained_features
    if df[feature].nunique(dropna=True) > 1
]

feature_positions = rng.choice(
    len(retained_features), size=5, replace=False
)

selected_features = [
    retained_features[i] for i in feature_positions
]

# Available Case-Analysis
available_results = []
for feature in selected_features:
    pass_values = df.loc[
        df["Outcome"] == "Pass", feature
    ].dropna()
    fail_values = df.loc[
        df["Outcome"] == "Fail", feature
    ].dropna()
    available_pass_mean = pass_values.mean()
    available_fail_mean = fail_values.mean()
    available_mean_diff = (available_pass_mean - available_fail_mean)

    available_results.append({
        "Feature": feature,
        "Pass count": len(pass_values),
        "Fail count": len(fail_values),
        "Pass mean": available_pass_mean,
        "Fail mean": available_fail_mean,
        "Available Mean Difference": available_mean_diff
    })
available_results_df = pd.DataFrame(available_results)

# Median Imputation
missing = np.isnan(raw_measurements)
observed = (~missing).sum(axis=0)

all_missing = observed == 0

observed_min = np.where(
    missing, np.inf, raw_measurements
).min(axis=0)

observed_max = np.where(
    missing, -np.inf, raw_measurements
).max(axis=0)

constant = (observed > 0) & (observed_min == observed_max)
median_df = df[selected_features].copy()
feature_medians = median_df.median()
imputed_df = median_df.fillna(feature_medians)


median_results = []
for i, feature in enumerate(selected_features):
    pass_values = imputed_df.loc[df["Outcome"] == "Pass", feature].to_numpy()
    fail_values = imputed_df.loc[df["Outcome"] == "Fail", feature].to_numpy()
    median_pass_mean = pass_values.mean()
    median_fail_mean = fail_values.mean()
    median_mean_diff = (median_pass_mean - median_fail_mean)
    median_results.append({
        "Feature": feature,
        "Pass count": len(pass_values),
        "Fail count": len(fail_values),
        "Pass mean": median_pass_mean,
        "Fail mean": median_fail_mean,
        "Median Mean Difference": median_mean_diff
    })
median_results_df = pd.DataFrame(median_results)

# Implement bootsrap resampling
bootstrap_reps = (200 + 10 * SEED_DIGIT)
def bootstrap_resampling(fail_values, pass_values, rng):
    bootstrap_differences = []
    
    for _ in range(bootstrap_reps):
        boot_fail = rng.choice(
            fail_values, size=len(fail_values), replace=True
        )
        boot_pass = rng.choice(
            pass_values, size=len(pass_values), replace=True
        )
        bootstrap_mean_diff = (np.mean(boot_pass) - np.mean(boot_fail))
        bootstrap_differences.append(bootstrap_mean_diff)
    bootstrap_differences = np.array(bootstrap_differences)
    bootstrap_mean = np.mean(bootstrap_differences)

    lower_ci, upper_ci = np.percentile(
        bootstrap_differences, 
        [2.5, 97.5]
    )
    return {

        "Bootstrap_Mean_Difference": bootstrap_mean,
        "CI Lower": lower_ci,
        "CI Upper": upper_ci
    }


results = []
for strategy in ["Available Case", "Median Imputation"]:
    for feature in selected_features:
        if strategy == "Available Case":
            fail_values = df.loc[
                df["Outcome"] == "Fail", feature
            ].dropna().to_numpy()

            pass_values = df.loc[
                df["Outcome"] == "Pass", feature
            ].dropna().to_numpy()

        else:

            fail_values = imputed_df.loc[
                df["Outcome"] == "Fail", feature
            ].to_numpy()

            pass_values = imputed_df.loc[
                df["Outcome"] == "Pass", feature
            ].to_numpy()

        result = bootstrap_resampling(
            fail_values,
            pass_values,
            rng
        )
        result["Feature"] = feature
        result["Strategy"] = strategy
        results.append(result)

results_df = pd.DataFrame(results)

available_comparison = available_results_df[
    ["Feature", "Available Mean Difference"]
]

median_comparison = median_results_df[
    ["Feature", "Median Mean Difference"]
]

bootstrap_comparison = results_df.pivot(
    index="Feature",
    columns="Strategy",
    values=[
        "Bootstrap_Mean_Difference",
        "CI Lower",
        "CI Upper"
    ]
)

# Flatten column names
bootstrap_comparison.columns = [
    f"{strategy} - {metric}"
    for metric, strategy in bootstrap_comparison.columns
]

bootstrap_comparison = bootstrap_comparison.reset_index()

# Combine all results
comparison = (
    available_comparison
    .merge(median_comparison, on="Feature")
    .merge(bootstrap_comparison, on="Feature")
)

comparison["Difference Between Mean Differences"] = (
    comparison["Median Mean Difference"]
    - comparison["Available Mean Difference"]
)

comparison["Available Direction"] = np.where(
    comparison["Available Mean Difference"] > 0,
    "Pass > Fail",
    np.where(
        comparison["Available Mean Difference"] < 0,
        "Pass < Fail",
        "Equal"
    )
)

comparison["Median Direction"] = np.where(
    comparison["Median Mean Difference"] > 0,
    "Pass > Fail",
    np.where(
        comparison["Median Mean Difference"] < 0,
        "Pass < Fail",
        "Equal"
    )
)

comparison["Direction Agreement"] = np.where(
    comparison["Available Direction"] ==
    comparison["Median Direction"],
    "Agree",
    "Disagree"
)

for strategy in ["Available Case", "Median Imputation"]:
    lower = comparison[f"{strategy} - CI Lower"]
    upper = comparison[f"{strategy} - CI Upper"]

    comparison[f"{strategy} CI Result"] = np.where(
        (lower > 0) | (upper < 0),
        "Excludes Zero",
        "Includes Zero"
    )
    
comparison["Available Rank"] = comparison["Available Mean Difference"].abs().rank(ascending=False, method="min").astype(int)
comparison["Median Rank"] = comparison["Median Mean Difference"].abs().rank(ascending=False, method="min").astype(int)
comparison["Rank Change"] = (comparison["Median Rank"] - comparison ["Available Rank"]).abs()

# FINAL COMPARISON TABLE

final_comparison = comparison[[
    "Feature",
    "Available Mean Difference",
    "Median Mean Difference",
    "Difference Between Mean Differences",
    "Available Case - Bootstrap_Mean_Difference",
    "Median Imputation - Bootstrap_Mean_Difference",
    "Available Direction",
    "Median Direction",
    "Direction Agreement",
    "Available Case - CI Lower",
    "Available Case - CI Upper",
    "Available Case CI Result",
    "Median Imputation - CI Lower",
    "Median Imputation - CI Upper",
    "Median Imputation CI Result",
    "Available Rank",
    "Median Rank",
    "Rank Change"
]]
print("--- ROBUST GROUPED STATISTICAL COMPARISON ---\n")
print(final_comparison.head())

final_comparison.to_csv(csv_path, index=False)

changed_sign_features = comparison.loc[
    comparison["Available Mean Difference"] == comparison["Median Mean Difference"], "Feature"
]
changed_sign = len(changed_sign_features) > 0

changed_rank_features = comparison.loc[
    comparison["Rank Change"] > 0, "Feature"
]
changed_rank = len(changed_rank_features) > 0

# NOTE: SENSITIVITY REPORT (TEXT FILE)
report = (
    "SECOM STATISTICAL ROBUSTNESS AND SENSITIVITY ANALYSIS\n"
    "Student-specific Parameters"
    f"Student: {LAST_NAME}\n"
    f"Student ID: {STUDENT_ID}\n"
    f"Random Seed: {RANDOM_SEED}\n"
    f"====================================\n"
    f"Selected Features: {selected_features}\n"
    f"Bootstrap Repititions: {bootstrap_reps}\n"
    f"Largest Change in Mean Difference: {round((comparison["Difference Between Mean Differences"].abs().max()), 4)}%\n"
    f"Features with Changed Difference Sign: {changed_sign_features if changed_sign else "No Changed Signs"}\n"
    f"Features with Changed Rank: {changed_rank_features if changed_rank else "No Changed Rank"}\n"
    f""
)

report_path.write_text(report, encoding="utf-8")
print("Comparison CSV Created.", csv_path.exists())
print("Report Text Created.", report_path.exists())
print("=" * 50)
print("Now running reproducibility verification... Please wait...")
print()




# REPRODUCIBILITY VERIFICATION
original_comparison = final_comparison.copy()
original_features = selected_features.copy()
rng = np.random.default_rng(RANDOM_SEED)
csv_path = OUTPUT_DIR / "robustness_comparison.csv"
report_path = OUTPUT_DIR / "sensitivity_report.txt"

n_features = raw_measurements.shape[1]
feature_names = [f"Feature_{i:03d}" for i in range(n_features)]
df = pd.DataFrame(
    raw_measurements.copy(), columns=feature_names
)

labels = np.loadtxt(label_path, usecols=0, dtype=int)

assert len(labels) == len(df)
assert np.isin(labels, [-1, 1]).all()

df["Result"] = labels
df["Outcome"] = np.where(
    labels == -1, "Pass", "Fail"
)

# Determining of observed measurements, and usable features generated by ChatGPT
pass_observed = (
    df.loc[df["Outcome"] == "Pass", feature_names]
    .count()
)

fail_observed = (
    df.loc[df["Outcome"] == "Fail", feature_names]
    .count()
)
# Find out the raw usable records and figure out the needed parameters.
retained_features = [
    feature for feature in feature_names
    if pass_observed[feature] >= 10
    and fail_observed[feature] >= 10
]

retained_features = [
    feature for feature in retained_features
    if df[feature].nunique(dropna=True) > 1
]

feature_positions = rng.choice(
    len(retained_features), size=5, replace=False
)

selected_features = [
    retained_features[i] for i in feature_positions
]



# Available Case-Analysis
available_results = []
for feature in selected_features:
    pass_values = df.loc[
        df["Outcome"] == "Pass", feature
    ].dropna()
    fail_values = df.loc[
        df["Outcome"] == "Fail", feature
    ].dropna()
    available_pass_mean = pass_values.mean()
    available_fail_mean = fail_values.mean()
    available_mean_diff = (available_pass_mean - available_fail_mean)

    available_results.append({
        "Feature": feature,
        "Pass count": len(pass_values),
        "Fail count": len(fail_values),
        "Pass mean": available_pass_mean,
        "Fail mean": available_fail_mean,
        "Available Mean Difference": available_mean_diff
    })
available_results_df = pd.DataFrame(available_results)

# Median Imputation
missing = np.isnan(raw_measurements)
observed = (~missing).sum(axis=0)

all_missing = observed == 0

observed_min = np.where(
    missing, np.inf, raw_measurements
).min(axis=0)

observed_max = np.where(
    missing, -np.inf, raw_measurements
).max(axis=0)

constant = (observed > 0) & (observed_min == observed_max)
median_df = df[selected_features].copy()
feature_medians = median_df.median()
imputed_df = median_df.fillna(feature_medians)


median_results = []
for i, feature in enumerate(selected_features):
    pass_values = imputed_df.loc[df["Outcome"] == "Pass", feature].to_numpy()
    fail_values = imputed_df.loc[df["Outcome"] == "Fail", feature].to_numpy()
    median_pass_mean = pass_values.mean()
    median_fail_mean = fail_values.mean()
    median_mean_diff = (median_pass_mean - median_fail_mean)
    median_results.append({
        "Feature": feature,
        "Pass count": len(pass_values),
        "Fail count": len(fail_values),
        "Pass mean": median_pass_mean,
        "Fail mean": median_fail_mean,
        "Median Mean Difference": median_mean_diff
    })
median_results_df = pd.DataFrame(median_results)

# Implement bootsrap resampling
bootstrap_reps = (200 + 10 * SEED_DIGIT)
def bootstrap_resampling(fail_values, pass_values, rng):
    bootstrap_differences = []
    
    for _ in range(bootstrap_reps):
        boot_fail = rng.choice(
            fail_values, size=len(fail_values), replace=True
        )
        boot_pass = rng.choice(
            pass_values, size=len(pass_values), replace=True
        )
        bootstrap_mean_diff = (np.mean(boot_pass) - np.mean(boot_fail))
        bootstrap_differences.append(bootstrap_mean_diff)
    bootstrap_differences = np.array(bootstrap_differences)
    bootstrap_mean = np.mean(bootstrap_differences)

    lower_ci, upper_ci = np.percentile(
        bootstrap_differences, 
        [2.5, 97.5]
    )
    return {

        "Bootstrap_Mean_Difference": bootstrap_mean,
        "CI Lower": lower_ci,
        "CI Upper": upper_ci
    }


results = []
for strategy in ["Available Case", "Median Imputation"]:
    for feature in selected_features:
        if strategy == "Available Case":
            fail_values = df.loc[
                df["Outcome"] == "Fail", feature
            ].dropna().to_numpy()

            pass_values = df.loc[
                df["Outcome"] == "Pass", feature
            ].dropna().to_numpy()

        else:

            fail_values = imputed_df.loc[
                df["Outcome"] == "Fail", feature
            ].to_numpy()

            pass_values = imputed_df.loc[
                df["Outcome"] == "Pass", feature
            ].to_numpy()

        result = bootstrap_resampling(
            fail_values,
            pass_values,
            rng
        )
        result["Feature"] = feature
        result["Strategy"] = strategy
        results.append(result)

results_df = pd.DataFrame(results)

available_comparison = available_results_df[
    ["Feature", "Available Mean Difference"]
]

median_comparison = median_results_df[
    ["Feature", "Median Mean Difference"]
]

bootstrap_comparison = results_df.pivot(
    index="Feature",
    columns="Strategy",
    values=[
        "Bootstrap_Mean_Difference",
        "CI Lower",
        "CI Upper"
    ]
)

# Flatten column names
bootstrap_comparison.columns = [
    f"{strategy} - {metric}"
    for metric, strategy in bootstrap_comparison.columns
]

bootstrap_comparison = bootstrap_comparison.reset_index()

# Combine all results
comparison = (
    available_comparison
    .merge(median_comparison, on="Feature")
    .merge(bootstrap_comparison, on="Feature")
)

comparison["Difference Between Mean Differences"] = (
    comparison["Median Mean Difference"]
    - comparison["Available Mean Difference"]
)

comparison["Available Direction"] = np.where(
    comparison["Available Mean Difference"] > 0,
    "Pass > Fail",
    np.where(
        comparison["Available Mean Difference"] < 0,
        "Pass < Fail",
        "Equal"
    )
)

comparison["Median Direction"] = np.where(
    comparison["Median Mean Difference"] > 0,
    "Pass > Fail",
    np.where(
        comparison["Median Mean Difference"] < 0,
        "Pass < Fail",
        "Equal"
    )
)

comparison["Direction Agreement"] = np.where(
    comparison["Available Direction"] ==
    comparison["Median Direction"],
    "Agree",
    "Disagree"
)

for strategy in ["Available Case", "Median Imputation"]:
    lower = comparison[f"{strategy} - CI Lower"]
    upper = comparison[f"{strategy} - CI Upper"]

    comparison[f"{strategy} CI Result"] = np.where(
        (lower > 0) | (upper < 0),
        "Excludes Zero",
        "Includes Zero"
    )
    
comparison["Available Rank"] = comparison["Available Mean Difference"].abs().rank(ascending=False, method="min").astype(int)
comparison["Median Rank"] = comparison["Median Mean Difference"].abs().rank(ascending=False, method="min").astype(int)
comparison["Rank Change"] = (comparison["Median Rank"] - comparison ["Available Rank"]).abs()

# FINAL COMPARISON TABLE

final_comparison = comparison[[
    "Feature",
    "Available Mean Difference",
    "Median Mean Difference",
    "Difference Between Mean Differences",
    "Available Case - Bootstrap_Mean_Difference",
    "Median Imputation - Bootstrap_Mean_Difference",
    "Available Direction",
    "Median Direction",
    "Direction Agreement",
    "Available Case - CI Lower",
    "Available Case - CI Upper",
    "Available Case CI Result",
    "Median Imputation - CI Lower",
    "Median Imputation - CI Upper",
    "Median Imputation CI Result",
    "Available Rank",
    "Median Rank",
    "Rank Change"
]]

print("--- ROBUST GROUPED STATISTICAL COMPARISON ---\n")
print(final_comparison.head())

final_comparison.to_csv(csv_path, index=False)

changed_sign_features = comparison.loc[
    comparison["Available Mean Difference"] == comparison["Median Mean Difference"], "Feature"
]
changed_sign = len(changed_sign_features) > 0

changed_rank_features = comparison.loc[
    comparison["Rank Change"] > 0, "Feature"
]
changed_rank = len(changed_rank_features) > 0

# REPRODUCIBILITY VERIFICATION
same_features = original_features == selected_features

same_results = original_comparison.equals(final_comparison)

print("=" * 50)
if same_features and same_results:
    print("Selected Features Match:", same_features)
    print("Statistical Results Match:", same_results)
    print("Program is REPRODUCIBLE? True")
else:
    print("Selected Features Match:", same_features)
    print("Statistical Results Match:", same_results)
    print("Program is Reproducible? False")
print("=" * 50)




--- ROBUST GROUPED STATISTICAL COMPARISON ---

       Feature  Available Mean Difference  Median Mean Difference  \
0  Feature_161                 334.953601              344.906449   
1  Feature_077                  -0.000677               -0.000724   
2  Feature_051                  -1.468797               -1.451028   
3  Feature_473                  -0.855168               -0.879680   
4  Feature_454                   0.409628                0.409296   

   Difference Between Mean Differences  \
0                             9.952848   
1                            -0.000046   
2                             0.017768   
3                            -0.024512   
4                            -0.000332   

   Available Case - Bootstrap_Mean_Difference  \
0                                  304.503542   
1                                   -0.000667   
2                                   -1.615331   
3                                   -0.692923   
4                                    0.4